In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv("water-treatment.data", header=None)
X_raw = data.apply(pd.to_numeric, errors="coerce")

# drop columns that are all NaN
X_raw = X_raw.drop(columns=X_raw.columns[X_raw.isna().all()])

# impute
X_raw = X_raw.fillna(X_raw.mean())

print("Shape:", X_raw.shape, "| Remaining NaNs:", X_raw.isna().sum().sum())

Shape: (527, 38) | Remaining NaNs: 0


In [ ]:
daily_diff = np.linalg.norm(X_raw.diff().fillna(0).values, axis=1)
thr = np.percentile(daily_diff, 90)  # prueba 85/90/95 luego
failure_day = (daily_diff > thr).astype(int)

print("Failure days:", int(failure_day.sum()))
n_days = len(failure_day)

Failure days: 53


In [ ]:
def make_early_warning_labels(failure_series, H):
    y = np.zeros(len(failure_series), dtype=np.float32)
    for t in range(len(failure_series)):
        if t + H < len(failure_series):
            y[t] = failure_series[t+1:t+H+1].max()
    return y

y_H1 = make_early_warning_labels(failure_day, 1)
y_H3 = make_early_warning_labels(failure_day, 3)
y_H7 = make_early_warning_labels(failure_day, 7)

print("Positives H1/H3/H7:", int(y_H1.sum()), int(y_H3.sum()), int(y_H7.sum()))

Positives H1/H3/H7: 53 130 237


In [ ]:
W = 14
X_values = X_raw.values.astype(np.float32)

X_seq = []
idx = []  # idx[i] = qué día real representa la secuencia i (día final de la ventana)

for t in range(W-1, n_days):
    X_seq.append(X_values[t-W+1:t+1])  # (W, C)
    idx.append(t)

X_seq = np.stack(X_seq)           # (N, W, C)
idx = np.array(idx, dtype=int)    # (N,)

# labels alineadas a secuencias
y1 = y_H1[W-1:].astype(np.float32)
y3 = y_H3[W-1:].astype(np.float32)
y7 = y_H7[W-1:].astype(np.float32)

N = X_seq.shape[0]
print("X_seq:", X_seq.shape, "| idx:", idx.shape, "| y:", len(y1))

X_seq: (514, 14, 38) | idx: (514,) | y: 514


In [ ]:
from sklearn.preprocessing import StandardScaler

train_end = int(0.6 * N)
val_end   = int(0.8 * N)

X_train = X_seq[:train_end]
X_val   = X_seq[train_end:val_end]
X_test  = X_seq[val_end:]

idx_train = idx[:train_end]
idx_val   = idx[train_end:val_end]
idx_test  = idx[val_end:]

def scale_sequences(X_train, X_val, X_test):
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

    def transform(X):
        X2 = X.reshape(-1, X.shape[-1])
        X2 = scaler.transform(X2)
        return X2.reshape(X.shape)

    return transform(X_train), transform(X_val), transform(X_test)

X_train_s, X_val_s, X_test_s = scale_sequences(X_train, X_val, X_test)

In [ ]:
def make_probs_all(n_days):
    return np.full(n_days, np.nan, dtype=float)

def fill_probs(probs_all, idx_part, probs_part):
    # idx_part: días reales, probs_part: prob de esos días
    for d, p in zip(idx_part, probs_part):
        probs_all[d] = float(p)
    return probs_all

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, W, C)
        self.y = torch.tensor(y, dtype=torch.float32)  # (N,)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        x = self.X[i].permute(1, 0)  # (C, W)
        return x, self.y[i]

def make_loaders(y_all, batch_size=64):
    y_train = y_all[:train_end]
    y_val   = y_all[train_end:val_end]
    y_test  = y_all[val_end:]

    train_ds = SeqDataset(X_train_s, y_train)
    val_ds   = SeqDataset(X_val_s, y_val)
    test_ds  = SeqDataset(X_test_s, y_test)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=False),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False),
        y_train, y_val, y_test
    )

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size): super().__init__(); self.chomp_size = chomp_size
    def forward(self, x): return x[:, :, :-self.chomp_size] if self.chomp_size > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.out_relu = nn.ReLU()

    def forward(self, x):
        out = self.drop1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.drop2(self.relu2(self.chomp2(self.conv2(out))))
        res = x if self.downsample is None else self.downsample(x)
        return self.out_relu(out + res)

class TCN(nn.Module):
    def __init__(self, n_inputs, channels=(32,32,32), kernel_size=3, dropout=0.1):
        super().__init__()
        layers=[]
        in_ch=n_inputs
        for i,out_ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation=2**i, dropout=dropout))
            in_ch=out_ch
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):          # x: (N, C, W)
        z = self.net(x)            # (N, hidden, W)
        z_last = z[:, :, -1]
        return self.head(z_last).squeeze(-1)  # logits

In [ ]:
def train_tcn_and_get_probs_all(y_all, epochs=60, lr=1e-3, batch_size=64, patience=10):
    train_loader, val_loader, test_loader, y_train, y_val, y_test = make_loaders(y_all, batch_size)

    # class imbalance weight on TRAIN
    pos = float(np.sum(y_train))
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(device)

    model = TCN(n_inputs=X_train_s.shape[-1], channels=(32,32,32), kernel_size=3, dropout=0.1).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_pr = -1.0
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        # early stopping on validation PR-AUC
        model.eval()
        val_probs = []
        val_true = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                probs = torch.sigmoid(model(xb)).cpu().numpy()
                val_probs.append(probs)
                val_true.append(yb.numpy())
        val_probs = np.concatenate(val_probs)
        val_true  = np.concatenate(val_true)
        val_pr = average_precision_score(val_true, val_probs) if val_true.sum() > 0 else 0.0

        if val_pr > best_val_pr + 1e-6:
            best_val_pr = val_pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # get probs for VAL and TEST (aligned to days via idx_val/idx_test)
    model.eval()
    probs_val = []
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(device)
            probs_val.append(torch.sigmoid(model(xb)).cpu().numpy())
    probs_val = np.concatenate(probs_val)

    probs_test = []
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            probs_test.append(torch.sigmoid(model(xb)).cpu().numpy())
    probs_test = np.concatenate(probs_test)

    probs_all = make_probs_all(n_days)
    probs_all = fill_probs(probs_all, idx_val, probs_val)
    probs_all = fill_probs(probs_all, idx_test, probs_test)

    return model, probs_all

In [ ]:
modelH3, probs_all_H3 = train_tcn_and_get_probs_all(y3)

In [ ]:
def extract_events(failure_day):
    return np.where(np.asarray(failure_day).astype(int) == 1)[0].tolist()

def build_pre_event_windows(event_days, H, n_days):
    windows=[]
    for t_f in event_days:
        s = max(0, t_f - H)
        e = t_f - 1
        if e >= s:
            windows.append((s,e,t_f))
    return windows

def mask_days_in_any_window(windows, n_days):
    m = np.zeros(n_days, dtype=bool)
    for s,e,_ in windows:
        m[s:e+1] = True
    return m

def event_based_metrics(probs_all, failure_day, H, threshold, day_range):
    probs_all = np.asarray(probs_all, dtype=float)
    failure_day = np.asarray(failure_day).astype(int)
    a,b = day_range
    a = max(0,a); b = min(len(failure_day)-1,b)

    event_days = [t for t in extract_events(failure_day) if a <= t <= b]
    windows = build_pre_event_windows(event_days, H, len(failure_day))
    # intersect windows with [a,b]
    windows_seg=[]
    for s,e,t_f in windows:
        s2=max(s,a); e2=min(e,b)
        if e2>=s2:
            windows_seg.append((s2,e2,t_f))

    alarm_days = [d for d in range(a,b+1) if (not np.isnan(probs_all[d])) and probs_all[d] >= threshold]
    pre_mask = mask_days_in_any_window(windows_seg, len(failure_day))

    # hit rate + lead time
    hits=0
    lead=[]
    for s,e,t_f in windows_seg:
        alarms_in = [d for d in alarm_days if s <= d <= e]
        if alarms_in:
            hits += 1
            first_alarm = min(alarms_in)
            lead.append(t_f - first_alarm)

    n_events = len(windows_seg)
    hit_rate = hits/n_events if n_events>0 else np.nan
    lead_mean = float(np.mean(lead)) if lead else np.nan
    lead_median = float(np.median(lead)) if lead else np.nan

    # false alarms per 30d
    false_alarm_days = [d for d in alarm_days if not pre_mask[d]]
    fa_per_30d = len(false_alarm_days) * (30.0 / (b-a+1))

    return {
        "n_events": n_events,
        "hit_rate": hit_rate,
        "lead_time_mean": lead_mean,
        "lead_time_median": lead_median,
        "false_alarms_per_30d": fa_per_30d,
        "n_alarm_days": len(alarm_days),
    }

def choose_threshold_on_val_robust(probs_all, failure_day, H, val_range,
                                   max_fa_per_30d_list=(1.0, 2.0, 4.0),
                                   min_hits=1):
    a,b = val_range
    val_probs = np.array([p for p in probs_all[a:b+1] if not np.isnan(p)], dtype=float)
    candidates = np.unique(np.quantile(val_probs, np.linspace(0.50, 0.995, 80)))
    candidates = np.sort(candidates)

    for max_fa in max_fa_per_30d_list:
        best_thr, best_metrics, best_hits = None, None, -1
        for thr in candidates:
            m = event_based_metrics(probs_all, failure_day, H, float(thr), val_range)
            hits = int(round(m["hit_rate"] * m["n_events"])) if not np.isnan(m["hit_rate"]) else 0
            if m["false_alarms_per_30d"] <= max_fa:
                if hits > best_hits:
                    best_hits = hits
                    best_thr = float(thr)
                    best_metrics = m
        if best_thr is not None and best_hits >= min_hits:
            return best_thr, best_metrics

    # fallback: el que minimiza false alarms
    thr = float(candidates[-1])
    return thr, event_based_metrics(probs_all, failure_day, H, thr, val_range)

In [ ]:
train_range = (idx_train[0], idx_train[-1])
val_range   = (idx_val[0], idx_val[-1])
test_range  = (idx_test[0], idx_test[-1])

print("Ranges (days):", train_range, val_range, test_range)

Ranges (days): (np.int64(13), np.int64(320)) (np.int64(321), np.int64(423)) (np.int64(424), np.int64(526))


In [ ]:
H = 3
thr, m_val = choose_threshold_on_val(probs_all_H3, failure_day, H, val_range, max_fa_per_30d=1.0)
m_test = event_based_metrics(probs_all_H3, failure_day, H, thr, test_range)

print("Chosen thr:", thr)
print("VAL:", m_val)
print("TEST:", m_test)

Chosen thr: 0.555265692185752
VAL: {'n_events': 13, 'hit_rate': 0.07692307692307693, 'lead_time_mean': 3.0, 'lead_time_median': 3.0, 'false_alarms_per_30d': np.float64(0.8737864077669902), 'n_alarm_days': 5}
TEST: {'n_events': 3, 'hit_rate': 0.0, 'lead_time_mean': nan, 'lead_time_median': nan, 'false_alarms_per_30d': np.float64(0.0), 'n_alarm_days': 0}


In [ ]:
modelH1, probs_all_H1 = train_tcn_and_get_probs_all(y1)
modelH7, probs_all_H7 = train_tcn_and_get_probs_all(y7)

for H, probs in [(1, probs_all_H1), (3, probs_all_H3), (7, probs_all_H7)]:
    thr, m_val = choose_threshold_on_val(probs, failure_day, H, val_range, max_fa_per_30d=1.0)
    m_test = event_based_metrics(probs, failure_day, H, thr, test_range)
    print(f"\nH={H} | thr={thr:.4f}")
    print("VAL :", m_val)
    print("TEST:", m_test)


H=1 | thr=0.6077
VAL : {'n_events': 13, 'hit_rate': 0.15384615384615385, 'lead_time_mean': 1.0, 'lead_time_median': 1.0, 'false_alarms_per_30d': np.float64(0.8737864077669902), 'n_alarm_days': 5}
TEST: {'n_events': 3, 'hit_rate': 0.0, 'lead_time_mean': nan, 'lead_time_median': nan, 'false_alarms_per_30d': np.float64(1.7475728155339805), 'n_alarm_days': 6}

H=3 | thr=0.5553
VAL : {'n_events': 13, 'hit_rate': 0.07692307692307693, 'lead_time_mean': 3.0, 'lead_time_median': 3.0, 'false_alarms_per_30d': np.float64(0.8737864077669902), 'n_alarm_days': 5}
TEST: {'n_events': 3, 'hit_rate': 0.0, 'lead_time_mean': nan, 'lead_time_median': nan, 'false_alarms_per_30d': np.float64(0.0), 'n_alarm_days': 0}

H=7 | thr=0.5228
VAL : {'n_events': 13, 'hit_rate': 0.8461538461538461, 'lead_time_mean': 3.8181818181818183, 'lead_time_median': 4.0, 'false_alarms_per_30d': np.float64(0.8737864077669902), 'n_alarm_days': 11}
TEST: {'n_events': 3, 'hit_rate': 0.6666666666666666, 'lead_time_mean': 6.5, 'lead_ti

# Rolling-origin evaluation (Event-base early warning)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

def set_seed(seed=123):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [ ]:
# Necesitas tener ya creados:
# X_seq: (N, W, C)
# idx:   (N,) días reales
# failure_day: (n_days,) 0/1
# y1,y3,y7: (N,) labels alineadas a X_seq

assert "X_seq" in globals()
assert "idx" in globals()
assert "failure_day" in globals()
assert "y1" in globals() and "y3" in globals() and "y7" in globals()

N, W, C = X_seq.shape
n_days = len(failure_day)

print("X_seq:", X_seq.shape)
print("idx range:", idx.min(), idx.max())
print("n_days:", n_days)
print("y sizes:", len(y1), len(y3), len(y7))

X_seq: (514, 14, 38)
idx range: 13 526
n_days: 527
y sizes: 514 514 514


In [ ]:
def extract_events(failure_day):
    return np.where(np.asarray(failure_day).astype(int) == 1)[0].tolist()

def build_pre_event_windows(event_days, H, n_days):
    windows=[]
    for t_f in event_days:
        s = max(0, t_f - H)
        e = t_f - 1
        if e >= s:
            windows.append((s,e,t_f))
    return windows

def mask_days_in_any_window(windows, n_days):
    m = np.zeros(n_days, dtype=bool)
    for s,e,_ in windows:
        m[s:e+1] = True
    return m

def event_based_metrics(probs_all, failure_day, H, threshold, day_range):
    probs_all = np.asarray(probs_all, dtype=float)
    failure_day = np.asarray(failure_day).astype(int)
    a,b = day_range
    a = max(0,a); b = min(len(failure_day)-1,b)

    event_days = [t for t in extract_events(failure_day) if a <= t <= b]
    windows = build_pre_event_windows(event_days, H, len(failure_day))

    # intersect windows with [a,b]
    windows_seg=[]
    for s,e,t_f in windows:
        s2=max(s,a); e2=min(e,b)
        if e2>=s2:
            windows_seg.append((s2,e2,t_f))

    alarm_days = [d for d in range(a,b+1) if (not np.isnan(probs_all[d])) and probs_all[d] >= threshold]
    pre_mask = mask_days_in_any_window(windows_seg, len(failure_day))

    # hit rate + lead time
    hits=0
    lead=[]
    for s,e,t_f in windows_seg:
        alarms_in = [d for d in alarm_days if s <= d <= e]
        if alarms_in:
            hits += 1
            first_alarm = min(alarms_in)
            lead.append(t_f - first_alarm)

    n_events = len(windows_seg)
    hit_rate = hits/n_events if n_events>0 else np.nan
    lead_mean = float(np.mean(lead)) if lead else np.nan
    lead_median = float(np.median(lead)) if lead else np.nan

    # false alarms per 30d
    false_alarm_days = [d for d in alarm_days if not pre_mask[d]]
    fa_per_30d = len(false_alarm_days) * (30.0 / (b-a+1))

    return {
        "n_events": n_events,
        "hit_rate": hit_rate,
        "lead_time_mean": lead_mean,
        "lead_time_median": lead_median,
        "false_alarms_per_30d": float(fa_per_30d),
        "n_alarm_days": len(alarm_days),
    }

def choose_threshold_on_val_op(probs_all, failure_day, H, val_range,
                               fa_budgets=(1.0, 2.0, 4.0),
                               min_hits=1,
                               min_alarm_days=1):
    a,b = val_range
    val_probs = np.array([p for p in probs_all[a:b+1] if not np.isnan(p)], dtype=float)
    if len(val_probs) == 0:
        raise ValueError("No probabilities in validation range")

    candidates = np.unique(np.quantile(val_probs, np.linspace(0.50, 0.995, 80)))
    candidates = np.sort(candidates)

    def count_alarm_days(range_ab, thr):
        aa,bb = range_ab
        return sum((not np.isnan(probs_all[d])) and (probs_all[d] >= thr) for d in range(aa,bb+1))

    for max_fa in fa_budgets:
        best = None  # (hits, -fa, thr, metrics)
        for thr in candidates:
            m = event_based_metrics(probs_all, failure_day, H, float(thr), val_range)
            hits = int(round(m["hit_rate"] * m["n_events"])) if not np.isnan(m["hit_rate"]) else 0
            alarm_days = count_alarm_days(val_range, float(thr))
            if m["false_alarms_per_30d"] <= max_fa and alarm_days >= min_alarm_days:
                key = (hits, -m["false_alarms_per_30d"], -float(thr))
                if (best is None) or (key > best[0]):
                    best = (key, float(thr), m, max_fa, alarm_days)
        if best is not None:
            _, thr_best, m_best, used_fa, alarm_days = best
            m_best = dict(m_best)
            m_best["used_fa_budget"] = used_fa
            m_best["val_alarm_days"] = alarm_days
            return thr_best, m_best

    # fallback
    thr = float(candidates[-1])
    m = event_based_metrics(probs_all, failure_day, H, thr, val_range)
    m["used_fa_budget"] = None
    m["val_alarm_days"] = 0
    return thr, m

In [ ]:
def rolling_folds_days(n_days, train_len=300, val_len=80, test_len=80, step=40):
    """
    Devuelve folds como rangos de días inclusivos: (train_range, val_range, test_range).
    """
    folds = []
    start_train = 0
    while True:
        tr = (start_train, start_train + train_len - 1)
        va = (tr[1] + 1, tr[1] + val_len)
        te = (va[1] + 1, va[1] + test_len)
        if te[1] >= n_days:
            break
        folds.append((tr, va, te))
        start_train += step
    return folds

# Ajusta longitudes si quieres (estos valores suelen dar 3-5 folds en 527 días)
folds = rolling_folds_days(n_days=n_days, train_len=260, val_len=80, test_len=80, step=30)
print("Num folds:", len(folds))
for k,(tr,va,te) in enumerate(folds):
    print(k, "train", tr, "val", va, "test", te)

Num folds: 4
0 train (0, 259) val (260, 339) test (340, 419)
1 train (30, 289) val (290, 369) test (370, 449)
2 train (60, 319) val (320, 399) test (400, 479)
3 train (90, 349) val (350, 429) test (430, 509)


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, W, C)
        self.y = torch.tensor(y, dtype=torch.float32)  # (N,)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        x = self.X[i].permute(1, 0)  # (C, W)
        return x, self.y[i]

def scale_by_train_fold(X_train, X_val, X_test):
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, X_train.shape[-1]))

    def transform(X):
        X2 = X.reshape(-1, X.shape[-1])
        X2 = scaler.transform(X2)
        return X2.reshape(X.shape)

    return transform(X_train), transform(X_val), transform(X_test)

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size] if self.chomp_size > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.out_relu = nn.ReLU()

    def forward(self, x):
        out = self.drop1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.drop2(self.relu2(self.chomp2(self.conv2(out))))
        res = x if self.downsample is None else self.downsample(x)
        return self.out_relu(out + res)

class TCN(nn.Module):
    def __init__(self, n_inputs, channels=(32,32,32), kernel_size=3, dropout=0.1):
        super().__init__()
        layers=[]
        in_ch=n_inputs
        for i,out_ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation=2**i, dropout=dropout))
            in_ch=out_ch
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):          # x: (N, C, W)
        z = self.net(x)            # (N, hidden, W)
        z_last = z[:, :, -1]
        return self.head(z_last).squeeze(-1)  # logits

In [ ]:
def probs_all_from_fold(model, loader, idx_part, n_days):
    model.eval()
    probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    probs = np.concatenate(probs)

    probs_all = np.full(n_days, np.nan, dtype=float)
    for d, p in zip(idx_part, probs):
        probs_all[int(d)] = float(p)
    return probs_all

def train_one_fold_tcn(X_train, y_train, X_val, y_val, X_test, y_test,
                       idx_val_days, idx_test_days,
                       epochs=60, lr=1e-3, batch_size=64, patience=10,
                       channels=(32,32,32), kernel_size=3, dropout=0.1):

    train_ds = SeqDataset(X_train, y_train)
    val_ds   = SeqDataset(X_val, y_val)
    test_ds  = SeqDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # imbalance weight
    pos = float(np.sum(y_train))
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(device)

    model = TCN(n_inputs=X_train.shape[-1], channels=channels, kernel_size=kernel_size, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss = np.inf
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

        # early stop on val loss (estable)
        model.eval()
        vlosses=[]
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vlosses.append(float(crit(model(xb), yb).item()))
        vloss = float(np.mean(vlosses))

        if vloss < best_val_loss - 1e-6:
            best_val_loss = vloss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # probs_all for val and test (separately, luego los combinamos)
    probs_all_val = probs_all_from_fold(model, val_loader, idx_val_days, n_days)
    probs_all_test = probs_all_from_fold(model, test_loader, idx_test_days, n_days)

    # combinar en un solo probs_all con val+test
    probs_all = np.full(n_days, np.nan, dtype=float)
    mask_val = ~np.isnan(probs_all_val)
    mask_te  = ~np.isnan(probs_all_test)
    probs_all[mask_val] = probs_all_val[mask_val]
    probs_all[mask_te]  = probs_all_test[mask_te]

    return probs_all

In [ ]:
def seq_indices_for_day_range(idx_days, day_range):
    a,b = day_range
    return np.where((idx_days >= a) & (idx_days <= b))[0]

def get_fold_data(X_seq, idx_days, y_all, tr_days, va_days, te_days):
    tr_i = seq_indices_for_day_range(idx_days, tr_days)
    va_i = seq_indices_for_day_range(idx_days, va_days)
    te_i = seq_indices_for_day_range(idx_days, te_days)

    X_tr, y_tr = X_seq[tr_i], y_all[tr_i]
    X_va, y_va = X_seq[va_i], y_all[va_i]
    X_te, y_te = X_seq[te_i], y_all[te_i]

    idx_va_days = idx_days[va_i]
    idx_te_days = idx_days[te_i]

    return (X_tr,y_tr),(X_va,y_va),(X_te,y_te), idx_va_days, idx_te_days

In [ ]:
def run_rolling_tcn_for_H(H, y_all, folds, seed_base=123, max_fa_per_30d=1.0,
                          epochs=60, lr=1e-3, batch_size=64, patience=10):
    rows=[]
    for k,(tr_days,va_days,te_days) in enumerate(folds):
        set_seed(seed_base + k)

        # extraer data fold
        (X_tr,y_tr),(X_va,y_va),(X_te,y_te), idx_va_days, idx_te_days = get_fold_data(
            X_seq, idx, y_all, tr_days, va_days, te_days
        )

        # scale (train only)
        X_tr_s, X_va_s, X_te_s = scale_by_train_fold(X_tr, X_va, X_te)

        # entrenar y sacar probs_all (val+test)
        probs_all = train_one_fold_tcn(
            X_tr_s, y_tr, X_va_s, y_va, X_te_s, y_te,
            idx_va_days, idx_te_days,
            epochs=epochs, lr=lr, batch_size=batch_size, patience=patience
        )

        # elegir umbral en val (robust)
        thr, m_val = choose_threshold_on_val_op(
            probs_all, failure_day, H, va_days,
            fa_budgets=(1.0, 2.0, 4.0),
            min_hits=1,
            min_alarm_days=1
        )

        m_test = event_based_metrics(probs_all, failure_day, H, thr, te_days)

        rows.append({
            "fold": k,
            "H": H,
            "train_days": tr_days,
            "val_days": va_days,
            "test_days": te_days,
            "thr": thr,
            "val_hit_rate": m_val["hit_rate"],
            "val_fa30": m_val["false_alarms_per_30d"],
            "val_lead_med": m_val["lead_time_median"],
            "test_hit_rate": m_test["hit_rate"],
            "test_fa30": m_test["false_alarms_per_30d"],
            "test_lead_med": m_test["lead_time_median"],
            "test_n_events": m_test["n_events"],
        })

        print(f"Fold {k} | H={H} | thr={thr:.3f} | test hit={m_test['hit_rate']:.3f} | test FA/30d={m_test['false_alarms_per_30d']:.2f} | test events={m_test['n_events']}")

    return pd.DataFrame(rows)

In [ ]:
df_roll_H3 = run_rolling_tcn_for_H(H=3, y_all=y3, folds=folds, max_fa_per_30d=1.0,
                                   epochs=60, lr=1e-3, batch_size=64, patience=10)
df_roll_H3

Fold 0 | H=3 | thr=0.580 | test hit=0.231 | test FA/30d=0.75 | test events=13
Fold 1 | H=3 | thr=0.555 | test hit=0.000 | test FA/30d=1.50 | test events=10
Fold 2 | H=3 | thr=0.653 | test hit=0.000 | test FA/30d=0.00 | test events=3
Fold 3 | H=3 | thr=0.562 | test hit=nan | test FA/30d=2.62 | test events=0


,fold,H,train_days,val_days,test_days,thr,val_hit_rate,val_fa30,val_lead_med,test_hit_rate,test_fa30,test_lead_med,test_n_events
0,0,3,"(0, 259)","(260, 339)","(340, 419)",0.579667,0.333333,0.75,1.5,0.230769,0.750,2.0,13
1,1,3,"(30, 289)","(290, 369)","(370, 449)",0.554536,0.428571,0.75,1.0,0.000000,1.500,NaN,10
2,2,3,"(60, 319)","(320, 399)","(400, 479)",0.653143,0.090909,0.00,1.0,0.000000,0.000,NaN,3
3,3,3,"(90, 349)","(350, 429)","(430, 509)",0.561527,0.230769,0.75,2.0,NaN,2.625,NaN,0


In [ ]:
def summarize_rolling(df):
    out = {}
    out["folds"] = len(df)
    out["test_hit_rate_mean"] = df["test_hit_rate"].mean()
    out["test_hit_rate_std"]  = df["test_hit_rate"].std(ddof=1)
    out["test_fa30_mean"]     = df["test_fa30"].mean()
    out["test_fa30_std"]      = df["test_fa30"].std(ddof=1)
    out["test_lead_med_mean"] = df["test_lead_med"].mean()
    out["test_lead_med_std"]  = df["test_lead_med"].std(ddof=1)
    out["test_events_total"]  = df["test_n_events"].sum()
    return out

summ_H3 = summarize_rolling(df_roll_H3)
summ_H3

{'folds': 4,
 'test_hit_rate_mean': np.float64(0.07692307692307693),
 'test_hit_rate_std': 0.13323467750529827,
 'test_fa30_mean': np.float64(1.21875),
 'test_fa30_std': 1.1197795541980573,
 'test_lead_med_mean': np.float64(2.0),
 'test_lead_med_std': nan,
 'test_events_total': np.int64(26)}

In [ ]:
df_roll_H1 = run_rolling_tcn_for_H(H=1, y_all=y1, folds=folds, max_fa_per_30d=1.0)
df_roll_H7 = run_rolling_tcn_for_H(H=7, y_all=y7, folds=folds, max_fa_per_30d=1.0)

summ_H1 = summarize_rolling(df_roll_H1)
summ_H7 = summarize_rolling(df_roll_H7)

summ_H1, summ_H7

Fold 0 | H=1 | thr=0.598 | test hit=0.000 | test FA/30d=1.12 | test events=13
Fold 1 | H=1 | thr=0.684 | test hit=0.000 | test FA/30d=0.38 | test events=10
Fold 2 | H=1 | thr=0.701 | test hit=0.000 | test FA/30d=0.00 | test events=3
Fold 3 | H=1 | thr=0.564 | test hit=nan | test FA/30d=1.88 | test events=0
Fold 0 | H=7 | thr=0.590 | test hit=0.385 | test FA/30d=0.75 | test events=13
Fold 1 | H=7 | thr=0.560 | test hit=0.600 | test FA/30d=2.62 | test events=10
Fold 2 | H=7 | thr=0.545 | test hit=0.333 | test FA/30d=0.00 | test events=3
Fold 3 | H=7 | thr=0.610 | test hit=nan | test FA/30d=1.12 | test events=0


({'folds': 4,
  'test_hit_rate_mean': np.float64(0.0),
  'test_hit_rate_std': 0.0,
  'test_fa30_mean': np.float64(0.84375),
  'test_fa30_std': 0.8315084184781294,
  'test_lead_med_mean': nan,
  'test_lead_med_std': nan,
  'test_events_total': np.int64(26)},
 {'folds': 4,
  'test_hit_rate_mean': np.float64(0.4393162393162393),
  'test_hit_rate_std': 0.14149881768598205,
  'test_fa30_mean': np.float64(1.125),
  'test_fa30_std': 1.103970108290981,
  'test_lead_med_mean': np.float64(3.0),
  'test_lead_med_std': 2.0,
  'test_events_total': np.int64(26)})

In [ ]:
def latex_summary_row(H, summ):
    def fmt(x):
        return f"{x:.3f}"
    return f"{H} & {fmt(summ['test_hit_rate_mean'])} $\\pm$ {fmt(summ['test_hit_rate_std'])} & " \
           f"{fmt(summ['test_fa30_mean'])} $\\pm$ {fmt(summ['test_fa30_std'])} & " \
           f"{fmt(summ['test_lead_med_mean'])} $\\pm$ {fmt(summ['test_lead_med_std'])} \\\\"

print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Rolling-origin event-based early-warning performance of the TCN model (mean $\pm$ SD across folds). Thresholds are selected on each fold's validation period to target $\leq$1 false alarm per 30 days.}")
print(r"\label{tab:rolling_event_based}")
print(r"\begin{tabular}{@{}lccc@{}}")
print(r"\toprule")
print(r"Horizon $H$ & Hit rate & False alarms / 30d & Lead time (median) \\")
print(r"\midrule")
print(latex_summary_row(3, summ_H3))
# si calculas H1 y H7, descomenta:
# print(latex_summary_row(1, summ_H1))
# print(latex_summary_row(7, summ_H7))
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

\begin{table}[ht]
\centering
\caption{Rolling-origin event-based early-warning performance of the TCN model (mean $\pm$ SD across folds). Thresholds are selected on each fold's validation period to target $\leq$1 false alarm per 30 days.}
\label{tab:rolling_event_based}
\begin{tabular}{@{}lccc@{}}
\toprule
Horizon $H$ & Hit rate & False alarms / 30d & Lead time (median) \\
\midrule
3 & 0.077 $\pm$ 0.133 & 1.219 $\pm$ 1.120 & 2.000 $\pm$ nan \\
\bottomrule
\end{tabular}
\end{table}


In [ ]:
print("Num folds:", len(folds))
for k,(tr,va,te) in enumerate(folds):
    tr_i = np.where((idx>=tr[0]) & (idx<=tr[1]))[0]
    va_i = np.where((idx>=va[0]) & (idx<=va[1]))[0]
    te_i = np.where((idx>=te[0]) & (idx<=te[1]))[0]
    n_te_events = int(np.sum((np.array(failure_day)==1) & (np.arange(len(failure_day))>=te[0]) & (np.arange(len(failure_day))<=te[1])))
    print(f"Fold {k}: seq train/val/test = {len(tr_i)}/{len(va_i)}/{len(te_i)} | test events={n_te_events} | ranges tr{tr} va{va} te{te}")


Num folds: 4
Fold 0: seq train/val/test = 247/80/80 | test events=13 | ranges tr(0, 259) va(260, 339) te(340, 419)
Fold 1: seq train/val/test = 260/80/80 | test events=11 | ranges tr(30, 289) va(290, 369) te(370, 449)
Fold 2: seq train/val/test = 260/80/80 | test events=3 | ranges tr(60, 319) va(320, 399) te(400, 479)
Fold 3: seq train/val/test = 260/80/80 | test events=0 | ranges tr(90, 349) va(350, 429) te(430, 509)


In [ ]:
# Debug fold 1 para H=3
k = 1
(tr_days,va_days,te_days) = folds[k]

(X_tr,y_tr),(X_va,y_va),(X_te,y_te), idx_va_days, idx_te_days = get_fold_data(X_seq, idx, y3, tr_days, va_days, te_days)
X_tr_s, X_va_s, X_te_s = scale_by_train_fold(X_tr, X_va, X_te)

set_seed(123 + k)
probs_all = train_one_fold_tcn(
    X_tr_s, y_tr, X_va_s, y_va, X_te_s, y_te,
    idx_va_days, idx_te_days
)

thr, m_val = choose_threshold_on_val_robust(probs_all, failure_day, 3, va_days, max_fa_per_30d_list=(1.0,2.0,4.0), min_hits=1)
m_test = event_based_metrics(probs_all, failure_day, 3, thr, te_days)

# Conteo de alarmas reales en test
a,b = te_days
alarm_days_test = [d for d in range(a,b+1) if (not np.isnan(probs_all[d])) and probs_all[d] >= thr]
print("thr:", thr)
print("VAL:", m_val)
print("TEST:", m_test)
print("Test alarm days:", len(alarm_days_test))
print("First 10 alarm days:", alarm_days_test[:10])


thr: 0.5545355832576752
VAL: {'n_events': 7, 'hit_rate': 0.42857142857142855, 'lead_time_mean': 1.3333333333333333, 'lead_time_median': 1.0, 'false_alarms_per_30d': 0.75, 'n_alarm_days': 4}
TEST: {'n_events': 10, 'hit_rate': 0.0, 'lead_time_mean': nan, 'lead_time_median': nan, 'false_alarms_per_30d': 1.5, 'n_alarm_days': 4}
Test alarm days: 4
First 10 alarm days: [399, 400, 427, 428]


In [ ]:
folds = rolling_folds_days(n_days=n_days, train_len=260, val_len=80, test_len=80, step=30)
print("Num folds:", len(folds))
for k,(tr,va,te) in enumerate(folds):
    n_te_events = int(np.sum((np.array(failure_day)==1) & (np.arange(len(failure_day))>=te[0]) & (np.arange(len(failure_day))<=te[1])))
    print(k, "test events:", n_te_events, "te", te)

Num folds: 4
0 test events: 13 te (340, 419)
1 test events: 11 te (370, 449)
2 test events: 3 te (400, 479)
3 test events: 0 te (430, 509)


In [ ]:
def count_events_in_range(failure_day, day_range):
    a,b = day_range
    fd = np.asarray(failure_day).astype(int)
    return int(fd[a:b+1].sum())

folds_valid = []
folds_info = []

for k,(tr,va,te) in enumerate(folds):
    n_te_events = count_events_in_range(failure_day, te)
    folds_info.append((k, tr, va, te, n_te_events))
    if n_te_events >= 1:
        folds_valid.append((tr,va,te))

print("All folds:")
for row in folds_info:
    print(f"Fold {row[0]} test events={row[4]} | te={row[3]}")

print("\nValid folds (test events >= 1):", len(folds_valid))
for k, tr, va, te, n in [r for r in folds_info if r[4] >= 1]:
    print(f"Fold {k} | test events={n} | te={te}")


All folds:
Fold 0 test events=13 | te=(340, 419)
Fold 1 test events=11 | te=(370, 449)
Fold 2 test events=3 | te=(400, 479)
Fold 3 test events=0 | te=(430, 509)

Valid folds (test events >= 1): 3
Fold 0 | test events=13 | te=(340, 419)
Fold 1 | test events=11 | te=(370, 449)
Fold 2 | test events=3 | te=(400, 479)


### y3

In [ ]:
df_roll_H3 = run_rolling_tcn_for_H(
    H=3, y_all=y3, folds=folds_valid,
    max_fa_per_30d=1.0,
    epochs=60, lr=1e-3, batch_size=64, patience=10
)
df_roll_H3


Fold 0 | H=3 | thr=0.580 | test hit=0.231 | test FA/30d=0.75 | test events=13
Fold 1 | H=3 | thr=0.555 | test hit=0.000 | test FA/30d=1.50 | test events=10
Fold 2 | H=3 | thr=0.653 | test hit=0.000 | test FA/30d=0.00 | test events=3


,fold,H,train_days,val_days,test_days,thr,val_hit_rate,val_fa30,val_lead_med,test_hit_rate,test_fa30,test_lead_med,test_n_events
0,0,3,"(0, 259)","(260, 339)","(340, 419)",0.579667,0.333333,0.75,1.5,0.230769,0.75,2.0,13
1,1,3,"(30, 289)","(290, 369)","(370, 449)",0.554536,0.428571,0.75,1.0,0.000000,1.50,NaN,10
2,2,3,"(60, 319)","(320, 399)","(400, 479)",0.653143,0.090909,0.00,1.0,0.000000,0.00,NaN,3


In [ ]:
df = df_roll_H3.copy()
print(df[["fold","thr","test_hit_rate","test_fa30","test_lead_med","test_n_events"]])

summ = {
    "folds": len(df),
    "test_events_total": int(df["test_n_events"].sum()),
    "hit_mean": float(df["test_hit_rate"].mean()),
    "hit_std": float(df["test_hit_rate"].std(ddof=1)) if len(df)>1 else 0.0,
    "fa_mean": float(df["test_fa30"].mean()),
    "fa_std": float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0,
    "lead_med_mean": float(df["test_lead_med"].mean(skipna=True)),
    "lead_med_std": float(df["test_lead_med"].std(ddof=1, skipna=True)) if len(df.dropna(subset=["test_lead_med"]))>1 else 0.0,
}
summ


   fold       thr  test_hit_rate  test_fa30  test_lead_med  test_n_events
0     0  0.579667       0.230769       0.75            2.0             13
1     1  0.554536       0.000000       1.50            NaN             10
2     2  0.653143       0.000000       0.00            NaN              3


{'folds': 3,
 'test_events_total': 26,
 'hit_mean': 0.07692307692307693,
 'hit_std': 0.13323467750529827,
 'fa_mean': 0.75,
 'fa_std': 0.75,
 'lead_med_mean': 2.0,
 'lead_med_std': 0.0}

In [ ]:
df = df_roll_H3.copy()

# hit_rate ponderado por nº de eventos (equivale a hits_totales / eventos_totales)
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

# false alarms / 30d: no es "por evento", así que lo dejo como media simple
fa_mean = float(df["test_fa30"].mean())

print("Total test events:", int(events_total))
print("Weighted hit rate:", hit_weighted)
print("Mean FA/30d:", fa_mean)


Total test events: 26
Weighted hit rate: 0.11538461538461539
Mean FA/30d: 0.75


In [ ]:
df = df_roll_H3.copy()

def fmt(x):
    if pd.isna(x): return "--"
    return f"{float(x):.3f}"

latex = []
latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}")
latex.append(r"\label{tab:rolling_H3_folds}")
latex.append(r"\begin{tabular}{@{}cccccc@{}}")
latex.append(r"\toprule")
latex.append(r"Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\")
latex.append(r"\midrule")
for _,r in df.iterrows():
    latex.append(f"{int(r['fold'])} & {fmt(r['thr'])} & {int(r['test_n_events'])} & {fmt(r['test_hit_rate'])} & {fmt(r['test_fa30'])} & {fmt(r['test_lead_med'])} \\\\")
latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

print("\n".join(latex))


\begin{table}[ht]
\centering
\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}
\label{tab:rolling_H3_folds}
\begin{tabular}{@{}cccccc@{}}
\toprule
Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\
\midrule
0 & 0.580 & 13 & 0.231 & 0.750 & 2.000 \\
1 & 0.555 & 10 & 0.000 & 1.500 & -- \\
2 & 0.653 & 3 & 0.000 & 0.000 & -- \\
\bottomrule
\end{tabular}
\end{table}


In [ ]:
df = df_roll_H3.copy()
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

fa_mean = float(df["test_fa30"].mean())
fa_std  = float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0

print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: " + str(int(events_total)) + r".}")
print(r"\label{tab:rolling_H3_summary}")
print(r"\begin{tabular}{@{}lc@{}}")
print(r"\toprule")
print(r"Metric & Value \\")
print(r"\midrule")
print(f"Event-weighted hit rate & {hit_weighted:.3f} \\\\")
print(f"False alarms / 30d & {fa_mean:.3f} $\\pm$ {fa_std:.3f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


\begin{table}[ht]
\centering
\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: 26.}
\label{tab:rolling_H3_summary}
\begin{tabular}{@{}lc@{}}
\toprule
Metric & Value \\
\midrule
Event-weighted hit rate & 0.115 \\
False alarms / 30d & 0.750 $\pm$ 0.750 \\
\bottomrule
\end{tabular}
\end{table}


## y1

In [ ]:
df_roll_H1 = run_rolling_tcn_for_H(
    H=1, y_all=y3, folds=folds_valid,
    max_fa_per_30d=1.0,
    epochs=60, lr=1e-3, batch_size=64, patience=10
)
df_roll_H1


Fold 0 | H=1 | thr=0.584 | test hit=0.077 | test FA/30d=1.12 | test events=13
Fold 1 | H=1 | thr=0.555 | test hit=0.000 | test FA/30d=1.50 | test events=10
Fold 2 | H=1 | thr=0.653 | test hit=0.000 | test FA/30d=0.00 | test events=3


,fold,H,train_days,val_days,test_days,thr,val_hit_rate,val_fa30,val_lead_med,test_hit_rate,test_fa30,test_lead_med,test_n_events
0,0,1,"(0, 259)","(260, 339)","(340, 419)",0.584403,0.250000,0.375,1.0,0.076923,1.125,1.0,13
1,1,1,"(30, 289)","(290, 369)","(370, 449)",0.554536,0.285714,0.750,1.0,0.000000,1.500,NaN,10
2,2,1,"(60, 319)","(320, 399)","(400, 479)",0.653143,0.090909,0.000,1.0,0.000000,0.000,NaN,3


In [ ]:
df = df_roll_H1.copy()
print(df[["fold","thr","test_hit_rate","test_fa30","test_lead_med","test_n_events"]])

summ = {
    "folds": len(df),
    "test_events_total": int(df["test_n_events"].sum()),
    "hit_mean": float(df["test_hit_rate"].mean()),
    "hit_std": float(df["test_hit_rate"].std(ddof=1)) if len(df)>1 else 0.0,
    "fa_mean": float(df["test_fa30"].mean()),
    "fa_std": float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0,
    "lead_med_mean": float(df["test_lead_med"].mean(skipna=True)),
    "lead_med_std": float(df["test_lead_med"].std(ddof=1, skipna=True)) if len(df.dropna(subset=["test_lead_med"]))>1 else 0.0,
}
summ


   fold       thr  test_hit_rate  test_fa30  test_lead_med  test_n_events
0     0  0.584403       0.076923      1.125            1.0             13
1     1  0.554536       0.000000      1.500            NaN             10
2     2  0.653143       0.000000      0.000            NaN              3


{'folds': 3,
 'test_events_total': 26,
 'hit_mean': 0.025641025641025644,
 'hit_std': 0.04441155916843275,
 'fa_mean': 0.875,
 'fa_std': 0.7806247497997998,
 'lead_med_mean': 1.0,
 'lead_med_std': 0.0}

In [ ]:
df = df_roll_H1.copy()

# hit_rate ponderado por nº de eventos (equivale a hits_totales / eventos_totales)
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

# false alarms / 30d: no es "por evento", así que lo dejo como media simple
fa_mean = float(df["test_fa30"].mean())

print("Total test events:", int(events_total))
print("Weighted hit rate:", hit_weighted)
print("Mean FA/30d:", fa_mean)


Total test events: 26
Weighted hit rate: 0.038461538461538464
Mean FA/30d: 0.875


In [ ]:
df = df_roll_H1.copy()

def fmt(x):
    if pd.isna(x): return "--"
    return f"{float(x):.3f}"

latex = []
latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}")
latex.append(r"\label{tab:rolling_H1_folds}")
latex.append(r"\begin{tabular}{@{}cccccc@{}}")
latex.append(r"\toprule")
latex.append(r"Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\")
latex.append(r"\midrule")
for _,r in df.iterrows():
    latex.append(f"{int(r['fold'])} & {fmt(r['thr'])} & {int(r['test_n_events'])} & {fmt(r['test_hit_rate'])} & {fmt(r['test_fa30'])} & {fmt(r['test_lead_med'])} \\\\")
latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

print("\n".join(latex))


\begin{table}[ht]
\centering
\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}
\label{tab:rolling_H1_folds}
\begin{tabular}{@{}cccccc@{}}
\toprule
Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\
\midrule
0 & 0.584 & 13 & 0.077 & 1.125 & 1.000 \\
1 & 0.555 & 10 & 0.000 & 1.500 & -- \\
2 & 0.653 & 3 & 0.000 & 0.000 & -- \\
\bottomrule
\end{tabular}
\end{table}


In [ ]:
df = df_roll_H1.copy()
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

fa_mean = float(df["test_fa30"].mean())
fa_std  = float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0

print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: " + str(int(events_total)) + r".}")
print(r"\label{tab:rolling_H3_summary}")
print(r"\begin{tabular}{@{}lc@{}}")
print(r"\toprule")
print(r"Metric & Value \\")
print(r"\midrule")
print(f"Event-weighted hit rate & {hit_weighted:.3f} \\\\")
print(f"False alarms / 30d & {fa_mean:.3f} $\\pm$ {fa_std:.3f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


\begin{table}[ht]
\centering
\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: 26.}
\label{tab:rolling_H3_summary}
\begin{tabular}{@{}lc@{}}
\toprule
Metric & Value \\
\midrule
Event-weighted hit rate & 0.038 \\
False alarms / 30d & 0.875 $\pm$ 0.781 \\
\bottomrule
\end{tabular}
\end{table}


## y7

In [ ]:
df_roll_H7 = run_rolling_tcn_for_H(
    H=7, y_all=y7, folds=folds_valid,
    max_fa_per_30d=1.0,
    epochs=60, lr=1e-3, batch_size=64, patience=10
)
df_roll_H7


Fold 0 | H=7 | thr=0.590 | test hit=0.385 | test FA/30d=0.75 | test events=13
Fold 1 | H=7 | thr=0.560 | test hit=0.600 | test FA/30d=2.62 | test events=10
Fold 2 | H=7 | thr=0.545 | test hit=0.333 | test FA/30d=0.00 | test events=3


,fold,H,train_days,val_days,test_days,thr,val_hit_rate,val_fa30,val_lead_med,test_hit_rate,test_fa30,test_lead_med,test_n_events
0,0,7,"(0, 259)","(260, 339)","(340, 419)",0.590496,0.416667,0.375,2.0,0.384615,0.750,3.0,13
1,1,7,"(30, 289)","(290, 369)","(370, 449)",0.560066,0.428571,0.750,1.0,0.600000,2.625,5.0,10
2,2,7,"(60, 319)","(320, 399)","(400, 479)",0.545120,0.363636,0.375,2.5,0.333333,0.000,1.0,3


In [ ]:
df = df_roll_H7.copy()
print(df[["fold","thr","test_hit_rate","test_fa30","test_lead_med","test_n_events"]])

summ = {
    "folds": len(df),
    "test_events_total": int(df["test_n_events"].sum()),
    "hit_mean": float(df["test_hit_rate"].mean()),
    "hit_std": float(df["test_hit_rate"].std(ddof=1)) if len(df)>1 else 0.0,
    "fa_mean": float(df["test_fa30"].mean()),
    "fa_std": float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0,
    "lead_med_mean": float(df["test_lead_med"].mean(skipna=True)),
    "lead_med_std": float(df["test_lead_med"].std(ddof=1, skipna=True)) if len(df.dropna(subset=["test_lead_med"]))>1 else 0.0,
}
summ


   fold       thr  test_hit_rate  test_fa30  test_lead_med  test_n_events
0     0  0.590496       0.384615      0.750            3.0             13
1     1  0.560066       0.600000      2.625            5.0             10
2     2  0.545120       0.333333      0.000            1.0              3


{'folds': 3,
 'test_events_total': 26,
 'hit_mean': 0.4393162393162393,
 'hit_std': 0.14149881768598205,
 'fa_mean': 1.125,
 'fa_std': 1.352081728298996,
 'lead_med_mean': 3.0,
 'lead_med_std': 2.0}

In [ ]:
df = df_roll_H7.copy()

# hit_rate ponderado por nº de eventos (equivale a hits_totales / eventos_totales)
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

# false alarms / 30d: no es "por evento", así que lo dejo como media simple
fa_mean = float(df["test_fa30"].mean())

print("Total test events:", int(events_total))
print("Weighted hit rate:", hit_weighted)
print("Mean FA/30d:", fa_mean)


Total test events: 26
Weighted hit rate: 0.46153846153846156
Mean FA/30d: 1.125


In [ ]:
df = df_roll_H7.copy()

def fmt(x):
    if pd.isna(x): return "--"
    return f"{float(x):.3f}"

latex = []
latex.append(r"\begin{table}[ht]")
latex.append(r"\centering")
latex.append(r"\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}")
latex.append(r"\label{tab:rolling_H7_folds}")
latex.append(r"\begin{tabular}{@{}cccccc@{}}")
latex.append(r"\toprule")
latex.append(r"Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\")
latex.append(r"\midrule")
for _,r in df.iterrows():
    latex.append(f"{int(r['fold'])} & {fmt(r['thr'])} & {int(r['test_n_events'])} & {fmt(r['test_hit_rate'])} & {fmt(r['test_fa30'])} & {fmt(r['test_lead_med'])} \\\\")
latex.append(r"\bottomrule")
latex.append(r"\end{tabular}")
latex.append(r"\end{table}")

print("\n".join(latex))


\begin{table}[ht]
\centering
\caption{Rolling-origin evaluation results for $H=3$ days. Thresholds are selected per fold on the validation period to target $\leq$1 false alarm per 30 days.}
\label{tab:rolling_H7_folds}
\begin{tabular}{@{}cccccc@{}}
\toprule
Fold & Threshold & Events & Hit rate & False alarms / 30d & Lead time (median) \\
\midrule
0 & 0.590 & 13 & 0.385 & 0.750 & 3.000 \\
1 & 0.560 & 10 & 0.600 & 2.625 & 5.000 \\
2 & 0.545 & 3 & 0.333 & 0.000 & 1.000 \\
\bottomrule
\end{tabular}
\end{table}


In [ ]:
df = df_roll_H7.copy()
hits_total = float((df["test_hit_rate"] * df["test_n_events"]).sum())
events_total = float(df["test_n_events"].sum())
hit_weighted = hits_total / events_total if events_total > 0 else np.nan

fa_mean = float(df["test_fa30"].mean())
fa_std  = float(df["test_fa30"].std(ddof=1)) if len(df)>1 else 0.0

print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: " + str(int(events_total)) + r".}")
print(r"\label{tab:rolling_H3_summary}")
print(r"\begin{tabular}{@{}lc@{}}")
print(r"\toprule")
print(r"Metric & Value \\")
print(r"\midrule")
print(f"Event-weighted hit rate & {hit_weighted:.3f} \\\\")
print(f"False alarms / 30d & {fa_mean:.3f} $\\pm$ {fa_std:.3f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")


\begin{table}[ht]
\centering
\caption{Rolling-origin event-based early-warning performance for $H=3$ days. The hit rate is reported as an event-weighted estimate (total detected events divided by total events) across folds. False alarms per 30 days are averaged across folds (mean $\pm$ SD). Total evaluated test events: 26.}
\label{tab:rolling_H3_summary}
\begin{tabular}{@{}lc@{}}
\toprule
Metric & Value \\
\midrule
Event-weighted hit rate & 0.462 \\
False alarms / 30d & 1.125 $\pm$ 1.352 \\
\bottomrule
\end{tabular}
\end{table}
